# Meridiano · Ciclo semanal del churn de anunciantes (martes)
Todo se ejecuta sobre las **tablas** (no hay CRM). Cada martes se calcula el riesgo con corte el lunes anterior y se proponen 4 cuentas nuevas por comercial. Ejecuta las celdas en orden. Las celdas marcadas como opcionales solo se usan si tienes esos archivos.

In [ ]:
# 1. Subir el zip del paquete
from google.colab import files
subido = files.upload(); nombre = list(subido)[0]
!rm -rf /content/meridiano-churn-produccion
!unzip -q -o "{nombre}" -d /content/
%cd /content/meridiano-churn-produccion
!git log --oneline

In [ ]:
# 2. Instalar dependencias
!pip install -q -r requirements.txt

In [ ]:
# 3. Tests (evaluación, propuestas y ciclo completo)
!cd evaluacion && python -m pytest -q
!cd propuestas && python -m pytest -q
!python -m pytest -q tests

In [ ]:
# 4. OPCIONAL · subir el histórico de ciclos anteriores (confirmaciones y cuentas propuestas, para el enfriamiento de 90 días)
# up = files.upload()   # confirmaciones_historico.csv y propuestas_historico.csv
# import shutil; [shutil.move(n, f'historico/{n}') for n in up]

In [ ]:
# 5. OPCIONAL · subir el plan devuelto por los comerciales (con las columnas de confirmación rellenas)
PLAN_DEVUELTO = None
# up = files.upload(); PLAN_DEVUELTO = list(up)[0]

In [ ]:
# 6. Ejecutar el ciclo de la semana (fecha de corte: el lunes)
FECHA_CORTE = '2026-09-28'
extra = f'--plan_devuelto "{PLAN_DEVUELTO}"' if PLAN_DEVUELTO else ''
!python ciclo_semanal.py --fecha_corte {FECHA_CORTE} {extra}

In [ ]:
# 7. Ver el cuadro de mando y el plan
import pandas as pd
pd.set_option('display.max_colwidth', 80)
display(pd.read_excel(f'salidas/{FECHA_CORTE}/cuadro_de_mando.xlsx', sheet_name='Resumen'))
plan = pd.read_excel(f'salidas/{FECHA_CORTE}/propuestas_retencion.xlsx', sheet_name='Plan_por_comercial')
display(plan[['comercial_nombre','orden_comercial','razon_social','tipo_riesgo','facturacion_en_riesgo','motivo_principal','accion_id','fecha_limite']])

## Demostración (simulación)
No hay comerciales reales: esta celda ejecuta tres ciclos semanales en dic-2025 (12 cuentas cada uno, sin repetir), **simula** las confirmaciones de cada semana con lo que pasó en las tablas y ejecuta el ciclo del 28-sep-2026 con esas 36 confirmaciones. Demuestra el circuito, **no valida E2** (el acuerdo depende del acierto que se elija).

In [ ]:
# 8. DEMO · circuito completo con confirmaciones simuladas
ACIERTO = 0.8   # prueba también 0.4 para ver la vuelta automática a la regla A
!rm -f historico/confirmaciones_historico.csv historico/propuestas_historico.csv
devuelto = ''
for i, f in enumerate(['2025-12-01', '2025-12-08', '2025-12-15']):
    !python ciclo_semanal.py --fecha_corte {f} {devuelto}
    !python simular_confirmaciones.py --plan salidas/{f}/propuestas_retencion.xlsx --acierto {ACIERTO} --salida sim_{i}.xlsx
    devuelto = f'--plan_devuelto sim_{i}.xlsx'
!python ciclo_semanal.py --fecha_corte 2026-09-28 {devuelto}
print(open('salidas/2026-09-28/estado_etiqueta.json').read())

In [ ]:
# 9. Descargar resultados e histórico (Colab no guarda nada entre sesiones)
!zip -q -r resultados_ciclo.zip salidas historico
files.download('resultados_ciclo.zip')